## Trial - Stent + Artery placement

Load stent skeleton data, generate a matching straight artery from the stent features, visualise each individually, then show them together with coinciding centrelines.

### Load stent data & set material parameters

In [89]:
import json, ast
import numpy as np
import pandas as pd
import pathlib
import trimesh

# ─── Stent selection ──────────────────────────────────────────────────────────
STENT_NAME = "stent2"
# STENT_NAME = "2crownCrimpedXienceStent"
# STENT_NAME = "16crownCrimpedXienceStent+extremaSupports"

# ─── Material parameters ──────────────────────────────────────────────────────
# Unit system: mm – N – tonne  (consistent with stent geometry in mm)
#   Stress / modulus  →  MPa  =  N / mm²
#   Density           →  t / mm³  (1 t/mm³ = 10⁹ kg/m³)
#
# Preset reference values:
#   Material             E [MPa]    nu [-]  rho [t/mm³]   eps_max [-]
#   Nitinol (NiTi)       50 000     0.33    6.45e-9        0.08   (superelastic)
#   Stainless steel 316L 200 000    0.30    7.90e-9        0.002
#   Cobalt-chromium      230 000    0.30    8.90e-9        0.003

MATERIAL_NAME      = "Nitinol"
YOUNGS_MODULUS     = 50_000.0   # [MPa]   Young's modulus
POISSONS_RATIO     = 0.33       # [-]     Poisson's ratio
DENSITY            = 6.45e-9   # [t/mm³] mass density
MAX_ELASTIC_STRAIN = 0.08       # [-]     max recoverable strain before permanent deformation

# ─── Load stent data ──────────────────────────────────────────────────────────
STENT_DIR = pathlib.Path("../../notebook_outputs") / STENT_NAME
print(f"Loading stent data from: {STENT_DIR.resolve()}")

with open(STENT_DIR / "stent_features.json") as f:
    features = json.load(f)
with open(STENT_DIR / "stent_centerline_direction.json") as f:
    cl_direction = np.array(json.load(f))

skel = pd.read_csv(STENT_DIR / "skeleton_points.csv")
skel["neighbor_ids"] = skel["neighbor_ids"].apply(ast.literal_eval)

# ─── Extend features with material parameters ──────────────────────────────────
features.update({
    "material_name":      MATERIAL_NAME,
    "youngs_modulus":     YOUNGS_MODULUS,                                   # [MPa]
    "poissons_ratio":     POISSONS_RATIO,                                   # [-]
    "shear_modulus":      YOUNGS_MODULUS / (2.0 * (1.0 + POISSONS_RATIO)), # [MPa] derived
    "density":            DENSITY,                                          # [t/mm³]
    "max_elastic_strain": MAX_ELASTIC_STRAIN,                               # [-]
})

# ─── Print summary ────────────────────────────────────────────────────────────
_GEO = {"length", "diameter", "radius", "strut_thickness",
        "z_min", "z_max", "r_inner", "r_outer", "r_mid",
        "center_cylinder_radius", "num_points", "n_regions", "conn_radius_3d"}
_MAT_ORDER = ("material_name", "youngs_modulus", "poissons_ratio", "shear_modulus",
              "density", "max_elastic_strain")
_MAT_UNITS = {
    "material_name":      "",
    "youngs_modulus":     "MPa",
    "poissons_ratio":     "-",
    "shear_modulus":      "MPa",
    "density":            "t/mm³",
    "max_elastic_strain": "-",
}

print("\n=== Geometry features ===")
for k, v in features.items():
    if k in _GEO:
        print(f"  {k:28s}: {v:.6g}" if isinstance(v, float) else f"  {k:28s}: {v}")
print(f"  {'centerline_direction':28s}: {cl_direction.round(6)}")

print("\n=== Material / simulation parameters ===")
for k in _MAT_ORDER:
    v    = features[k]
    unit = _MAT_UNITS.get(k, "")
    val  = f"{v:.6g}" if isinstance(v, float) else str(v)
    print(f"  {k:28s}: {val:<16s}  [{unit}]" if unit else f"  {k:28s}: {val}")

print(f"\nSkeleton nodes : {len(skel):,}")

Loading stent data from: /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/notebook_outputs/stent2

=== Geometry features ===
  length                      : 18.0936
  diameter                    : 3.15063
  radius                      : 1.57531
  strut_thickness             : 0.106733
  z_min                       : -9.04029
  z_max                       : 9.05326
  r_inner                     : 1.46555
  r_outer                     : 1.57228
  r_mid                       : 1.51892
  center_cylinder_radius      : 0.734348
  num_points                  : 1000000
  n_regions                   : 130
  conn_radius_3d              : 0.0394311
  centerline_direction        : [-8.10e-05 -1.00e+00 -1.03e-04]

=== Material / simulation parameters ===
  material_name               : Nitinol
  youngs_modulus              : 50000             [MPa]
  poissons_ratio              : 0.33              [-]
  shear_modulus               : 18797             [MPa]
  density                     : 6

### Visualise stent skeleton (trimesh — interactive)

In [90]:
# Build unique edge segments from neighbor_ids
coords_idx = skel.set_index("skeleton_point_id")[["x", "y", "z"]]
segments = []
seen = set()
for _, row in skel.iterrows():
    pid = int(row["skeleton_point_id"])
    p0 = coords_idx.loc[pid].to_numpy()
    for nid in row["neighbor_ids"]:
        key = (min(pid, nid), max(pid, nid))
        if key in seen or nid not in coords_idx.index:
            continue
        seen.add(key)
        p1 = coords_idx.loc[nid].to_numpy()
        segments.append([p0, p1])
segments = np.array(segments)
print(f"Unique edges: {len(segments):,}")

# Separate node types for colouring
line_pts     = skel[skel["node_type"] == "line"    ][["x", "y", "z"]].to_numpy()
junction_pts = skel[skel["node_type"] == "junction"][["x", "y", "z"]].to_numpy()

pc_line     = trimesh.PointCloud(line_pts,     colors=np.tile([70,  130, 180, 120], (len(line_pts),     1)))
pc_junction = trimesh.PointCloud(junction_pts, colors=np.tile([255, 140,   0, 255], (len(junction_pts), 1)))

path = trimesh.load_path(segments)
path.colors = np.tile([100, 160, 220, 200], (len(path.entities), 1))

scene = trimesh.Scene([path, pc_line, pc_junction])
scene.show()

Unique edges: 21,166


### Generate matching straight artery

In [109]:
# ─── User parameters ──────────────────────────────────────────────────────────
ARTERY_TYPE      = "s_bend"  # "straight" | "curved" | "s_bend" | "tapered"
NOISE_AMPLITUDE  = 0.2 # fractional radius perturbation (0 = smooth, 0.05 = ±5%)
NOISE_SEED       = 42        # integer for reproducibility, or None for random

import sys
ARTERY_DIR = pathlib.Path("../../input_examples/artery_geometries").resolve()
if str(ARTERY_DIR) not in sys.path:
    sys.path.insert(0, str(ARTERY_DIR))

from generate_artery import (
    generate_straight_artery, generate_curved_artery,
    generate_s_bend_artery,   generate_tapered_artery,
    straight_centreline, curved_centreline, s_bend_centreline,
)

# Artery lumen radius: stent r_outer + 1 mm clearance
artery_radius = features["r_outer"] + 1
stent_length  = features["length"]

# ─── Compute safe bend radii (arc must fit inside artery length) ──────────────
# Angles MUST match the bend_angle_deg values used in ARTERY_CONFIGS below.
# curved : 1 arc  → need bend_radius < 0.9 * length / angle_rad
# s_bend : 2 arcs → need bend_radius < 0.9 * length / (2 * angle_rad)
_len_c = stent_length * 1.5
_len_s = stent_length * 2.0
_ba_c  = np.radians(20.0)                          # matches curved  bend_angle_deg=20
_ba_s  = np.radians(45.0)                          # matches s_bend  bend_angle_deg=45
_br_c  = min(20.0, 0.9 * _len_c / _ba_c)          # curved : 1 arc fits with 10% margin
_br_s  = min(15.0, 0.9 * _len_s / (2.0 * _ba_s))  # s_bend : 2 arcs fit with 10% margin

# ─── Per-type mesh parameters ─────────────────────────────────────────────────
ARTERY_CONFIGS = {
    "straight": dict(
        radius=artery_radius, length=stent_length * 1.5,
        n_circumference=64, n_axial=150,
    ),
    "curved": dict(
        radius=artery_radius, length=stent_length * 1.5,
        bend_radius=_br_c, bend_angle_deg=20.0,
        n_circumference=64, n_axial=150,
    ),
    "s_bend": dict(
        radius=artery_radius, length=stent_length * 2.0,
        bend_radius=_br_s, bend_angle_deg=45.0,
        n_circumference=64, n_axial=150,
    ),
    "tapered": dict(
        radius_proximal=artery_radius + 0.3, radius_distal=artery_radius - 0.2,
        length=stent_length * 1.5,
        n_circumference=64, n_axial=150,
    ),
}
MESH_GEN = {
    "straight": generate_straight_artery,
    "curved":   generate_curved_artery,
    "s_bend":   generate_s_bend_artery,
    "tapered":  generate_tapered_artery,
}

p = ARTERY_CONFIGS[ARTERY_TYPE]
artery_mesh = MESH_GEN[ARTERY_TYPE](
    **p, noise_amplitude=NOISE_AMPLITUDE, noise_seed=NOISE_SEED
)

# ─── Extract centreline (same parameterisation as mesh builder) ───────────────
if ARTERY_TYPE in ("straight", "tapered"):
    artery_cl = straight_centreline(p["length"], p["n_axial"])
elif ARTERY_TYPE == "curved":
    artery_cl = curved_centreline(p["length"], p["bend_radius"], p["bend_angle_deg"], p["n_axial"])
elif ARTERY_TYPE == "s_bend":
    artery_cl = s_bend_centreline(p["length"], p["bend_radius"], p["bend_angle_deg"], p["n_axial"])

total_arc = np.linalg.norm(np.diff(artery_cl, axis=0), axis=1).sum()

print(f"Artery type      : {ARTERY_TYPE}")
print(f"Artery radius    : {artery_radius:.3f} mm")
print(f"Noise amplitude  : {NOISE_AMPLITUDE} ({NOISE_AMPLITUDE*100:.0f}% of radius)  seed={NOISE_SEED}")
if ARTERY_TYPE == "curved":
    print(f"Bend radius      : {_br_c:.2f} mm  (arc = {_br_c * _ba_c:.2f} mm)")
if ARTERY_TYPE == "s_bend":
    print(f"Bend radius      : {_br_s:.2f} mm  (2× arc = {2 * _br_s * _ba_s:.2f} mm)")
print(f"Arc length       : {total_arc:.2f} mm  (stent {stent_length:.2f} mm = {stent_length/total_arc*100:.0f}% of artery)")
print(f"Centreline       : {len(artery_cl)} points  bounds {artery_cl.min(0).round(2)} → {artery_cl.max(0).round(2)}")
print(f"Mesh             : {len(artery_mesh.vertices):,} vertices  {len(artery_mesh.faces):,} faces  watertight={artery_mesh.is_watertight}")

Artery type      : s_bend
Artery radius    : 2.572 mm
Noise amplitude  : 0.2 (20% of radius)  seed=42
Bend radius      : 15.00 mm  (2× arc = 23.56 mm)
Arc length       : 36.19 mm  (stent 18.09 mm = 50% of artery)
Centreline       : 151 points  bounds [0. 0. 0.] → [11.76  0.   32.61]
Mesh             : 9,666 vertices  19,328 faces  watertight=True


### Visualise artery (trimesh — interactive)

In [110]:
artery_vis = artery_mesh.copy()
artery_vis.visual.face_colors = np.tile([210, 100, 90, 180], (len(artery_vis.faces), 1))

# Centreline as a connected gold polyline
cl_segs_vis = np.stack([artery_cl[:-1], artery_cl[1:]], axis=1)
cl_path_vis = trimesh.load_path(cl_segs_vis)
cl_path_vis.colors = np.tile([255, 215, 0, 255], (len(cl_path_vis.entities), 1))

scene_artery = trimesh.Scene([artery_vis, cl_path_vis])
scene_artery.show()

### Map stent skeleton onto artery centreline & combined scene (trimesh — interactive)

In [111]:
# ─── 1. Arc-length parameterisation of artery centreline ──────────────────────
seg_lens = np.linalg.norm(np.diff(artery_cl, axis=0), axis=1)
arc_lens  = np.concatenate([[0.0], np.cumsum(seg_lens)])
t_artery  = arc_lens / arc_lens[-1]          # normalised arc length ∈ [0, 1]

# ─── 2. Rotation-minimising frame (RMF) along artery centreline ───────────────
def compute_rmf(cl):
    n = len(cl)
    T      = np.zeros_like(cl, dtype=float)
    T[0]   = cl[1] - cl[0]
    T[-1]  = cl[-1] - cl[-2]
    T[1:-1]= cl[2:] - cl[:-2]
    nrm    = np.linalg.norm(T, axis=1, keepdims=True)
    T     /= np.where(nrm > 1e-12, nrm, 1.0)

    seed = np.array([1.0, 0.0, 0.0])
    if abs(T[0] @ seed) > 0.9:
        seed = np.array([0.0, 1.0, 0.0])
    N0  = np.cross(T[0], seed);  N0 /= np.linalg.norm(N0)

    N    = np.zeros_like(cl, dtype=float);  N[0] = N0
    for i in range(1, n):
        Ni = N[i-1] - (N[i-1] @ T[i]) * T[i]
        l  = np.linalg.norm(Ni)
        N[i] = Ni / l if l > 1e-12 else N[i-1]

    return T, N, np.cross(T, N)

_, N_arr, B_arr = compute_rmf(artery_cl)

# ─── 3. Centre stent within the artery arc ────────────────────────────────────
stent_arc_frac = features["length"] / arc_lens[-1]
t_start        = (1.0 - stent_arc_frac) / 2.0

# ─── 4. Vectorised mapping of skeleton nodes ──────────────────────────────────
z_min, z_max = features["z_min"], features["z_max"]
xyz      = skel[["x", "y", "z"]].values
t_stent  = (xyz[:, 2] - z_min) / (z_max - z_min)
t_a      = np.clip(t_start + t_stent * stent_arc_frac, 0.0, 1.0)

cl_m = np.column_stack([np.interp(t_a, t_artery, artery_cl[:, k]) for k in range(3)])
N_m  = np.column_stack([np.interp(t_a, t_artery, N_arr[:, k])     for k in range(3)])
B_m  = np.column_stack([np.interp(t_a, t_artery, B_arr[:, k])     for k in range(3)])
N_m /= np.linalg.norm(N_m, axis=1, keepdims=True)
B_m /= np.linalg.norm(B_m, axis=1, keepdims=True)

# new position = centreline point + x*N + y*B  (z offset already encoded in t_a)
skel_mapped = cl_m + xyz[:, 0:1] * N_m + xyz[:, 1:2] * B_m
print(f"Mapped {len(skel_mapped):,} skeleton nodes")

# ─── 5. Rebuild edge segments with mapped positions ───────────────────────────
# skeleton_point_id is sequential 0..N-1, so skel_mapped[nid] is direct
n_nodes = len(skel_mapped)
segs_m, seen_m = [], set()
for i, nbrs in enumerate(skel["neighbor_ids"]):
    pid = int(skel.iloc[i]["skeleton_point_id"])
    for nid in nbrs:
        if nid >= n_nodes:
            continue
        key = (min(pid, nid), max(pid, nid))
        if key in seen_m:
            continue
        seen_m.add(key)
        segs_m.append([skel_mapped[i], skel_mapped[nid]])
segs_m = np.array(segs_m)

line_mask = (skel["node_type"] == "line").values
junc_mask = (skel["node_type"] == "junction").values
line_pts_m = skel_mapped[line_mask]
junc_pts_m = skel_mapped[junc_mask]
print(f"Rebuilt {len(segs_m):,} edge segments")

# ─── 6. Combined scene ────────────────────────────────────────────────────────
# Artery: all mesh vertices as a semi-transparent point cloud
artery_cloud = trimesh.PointCloud(
    artery_mesh.vertices,
    colors=np.tile([220, 100, 90, 70], (len(artery_mesh.vertices), 1)),
)

# Artery centreline (gold polyline)
cl_path_comb = trimesh.load_path(np.stack([artery_cl[:-1], artery_cl[1:]], axis=1))
cl_path_comb.colors = np.tile([255, 215, 0, 255], (len(cl_path_comb.entities), 1))

# Mapped stent skeleton
path_m    = trimesh.load_path(segs_m)
path_m.colors = np.tile([100, 160, 220, 220], (len(path_m.entities), 1))
pc_line_m = trimesh.PointCloud(line_pts_m, colors=np.tile([70,  130, 180, 150], (len(line_pts_m), 1)))
pc_junc_m = trimesh.PointCloud(junc_pts_m, colors=np.tile([255, 140,   0, 255], (len(junc_pts_m), 1)))

scene_combined = trimesh.Scene([artery_cloud, cl_path_comb, path_m, pc_line_m, pc_junc_m])
scene_combined.show()

Mapped 20,968 skeleton nodes
Rebuilt 21,166 edge segments


### Geometric & material compatibility check

Before launching the expensive contact simulation, verify that the stent and artery are compatible.

| # | Check | Basis |
|---|---|---|
| 1 | **Length** | Stent arc ≤ 95% of artery arc |
| 2 | **Delivery** | Crimped OD < artery ID |
| 3 | **Containment** | Mapped skeleton nodes inside lumen |
| 4 | **Clearance** | Min wall distance ≥ strut radius |
| 5 | **Bending strain** | ε = r_strut / R_artery < ε_max (material) — or geometric heuristic if ε_max not set |

In [94]:
from scipy.spatial import cKDTree


def _inside_mesh_kdtree(mesh, points):
    """
    Point-in-closed-mesh test — no rtree required.

    Builds a KD-tree on mesh *vertices*, finds the nearest surface vertex for
    each query point, then checks the dot product of the vector
    (surface_vertex → query_point) with the outward vertex normal.

      dot < 0  →  inside  (vector opposes outward normal)
      dot ≥ 0  →  outside

    Returns
    -------
    inside : (N,) bool ndarray
    dists  : (N,) float ndarray — distance to nearest surface vertex [mm]
    """
    tree  = cKDTree(mesh.vertices)
    dists, vidx = tree.query(points, workers=-1)
    to_pt   = points - mesh.vertices[vidx]
    normals = mesh.vertex_normals[vidx]
    dot     = np.einsum("ij,ij->i", to_pt, normals)
    return dot < 0, dists


def check_stent_artery_fit(
    skel_mapped, skel, features, artery_mesh, artery_cl, artery_radius,
    n_sample=3000, rng_seed=42,
):
    """
    Five compatibility checks between a placed stent skeleton and an artery mesh.
    Check 5 (bending strain) is material-based when 'max_elastic_strain' is present
    in features, otherwise falls back to a geometric heuristic.

    Parameters
    ----------
    skel_mapped   : (N, 3) ndarray
    skel          : DataFrame  (skeleton table with node_type, neighbor_ids)
    features      : dict       (stent_features + material parameters from cell 2.1)
    artery_mesh   : trimesh.Trimesh
    artery_cl     : (M, 3) ndarray  artery centreline
    artery_radius : float           nominal artery lumen radius [mm]
    n_sample      : int             nodes sampled for checks 3 & 4

    Returns
    -------
    dict  {check_name: {passed, note, …values…}}
    """
    rng    = np.random.default_rng(rng_seed)
    checks = {}

    # ── 1. Length ──────────────────────────────────────────────────────────────
    seg_l = np.linalg.norm(np.diff(artery_cl, axis=0), axis=1)
    arc   = seg_l.sum()
    slen  = features["length"]
    fill  = slen / arc
    checks["length"] = dict(
        stent_length_mm = round(slen, 3),
        artery_arc_mm   = round(arc,  3),
        fill_fraction   = round(fill, 4),
        passed          = fill < 0.95,
        note = f"Stent {slen:.2f} mm fills {fill*100:.1f}% of {arc:.2f} mm artery arc",
    )

    # ── 2. Delivery feasibility ────────────────────────────────────────────────
    stent_od = 2.0 * features["r_outer"]
    artery_d = 2.0 * artery_radius
    fits     = stent_od < artery_d
    margin   = (artery_d - stent_od) / artery_d * 100
    checks["delivery"] = dict(
        crimped_OD_mm     = round(stent_od, 3),
        artery_ID_mm      = round(artery_d,  3),
        radial_margin_pct = round(margin, 1),
        passed            = fits,
        note = (
            f"Crimped OD {stent_od:.3f} mm < artery ID {artery_d:.3f} mm "
            f"({margin:.1f}% radial margin)"
            if fits else
            f"Crimped OD {stent_od:.3f} mm exceeds artery ID {artery_d:.3f} mm — delivery blocked"
        ),
    )

    # ── 3 & 4. Containment + wall clearance (shared KD-tree query) ────────────
    print("  [3/5] Containment + [4/5] Clearance … ", end="", flush=True)
    idx = rng.choice(len(skel_mapped), min(n_sample, len(skel_mapped)), replace=False)
    inside, dists = _inside_mesh_kdtree(artery_mesh, skel_mapped[idx])

    pct    = inside.mean() * 100
    checks["containment"] = dict(
        sampled    = len(idx),
        pct_inside = round(pct, 1),
        passed     = pct >= 95.0,
        note = f"{pct:.1f}% of {len(idx):,} sampled nodes inside artery lumen",
    )

    n_penet  = int((~inside).sum())
    clr      = np.where(inside, dists, -dists)
    min_cl   = float(clr.min())
    strut_r  = features.get("strut_thickness", 0.0) / 2.0
    passed_cl= (n_penet == 0) and (min_cl >= strut_r)
    print(f"{pct:.1f}% inside, min clearance {min_cl:.3f} mm, {n_penet} penetrating")
    checks["clearance"] = dict(
        min_clearance_mm = round(min_cl,  4),
        strut_radius_mm  = round(strut_r, 4),
        n_penetrating    = n_penet,
        sampled          = len(idx),
        passed           = passed_cl,
        note = (
            f"Min wall clearance {min_cl:.3f} mm "
            f"(strut radius {strut_r:.3f} mm), {n_penet} penetrating nodes"
        ),
    )

    # ── 5. Bending strain at artery curvature ─────────────────────────────────
    # Compute artery curvature from centreline second derivative
    T    = np.diff(artery_cl, axis=0)
    sl   = np.linalg.norm(T, axis=1)
    T_u  = T / sl[:, None]
    dT   = np.diff(T_u, axis=0)
    avg  = (sl[:-1] + sl[1:]) / 2.0
    kap  = np.linalg.norm(dT, axis=1) / avg
    kmax = float(kap.max()) if len(kap) else 0.0
    Rmin = (1.0 / kmax) if kmax > 1e-9 else float("inf")

    # Bending stiffness EI (always computed when geometry + material are available)
    E    = features.get("youngs_modulus")
    EI   = (E * np.pi * strut_r**4 / 4.0) if (E and strut_r > 0) else None

    eps_max = features.get("max_elastic_strain")

    if eps_max is not None and strut_r > 0:
        # ── Material-based: ε = r_strut / R  (Euler-Bernoulli outer-fiber strain) ──
        # Conservative upper bound — the full 3D stent distributes bending,
        # so actual per-strut strain is lower, but this is the standard screening check.
        eps_actual = (strut_r / Rmin) if np.isfinite(Rmin) else 0.0
        ok_curv    = eps_actual < eps_max
        if np.isfinite(Rmin):
            curv_note = (
                f"Max strut bending strain {eps_actual*100:.3f}% at R_min={Rmin:.1f} mm "
                f"({'<' if ok_curv else '≥'} elastic limit {eps_max*100:.1f}%)"
            )
        else:
            curv_note = "Artery is straight — zero bending strain"
            ok_curv   = True
        check_basis = f"material  [ε = r_strut/R,  limit = {eps_max*100:.1f}%]"
    else:
        # ── Geometric heuristic fallback ──────────────────────────────────────
        thresh      = slen / 2.0
        eps_actual  = None
        ok_curv     = Rmin > thresh
        if np.isfinite(Rmin):
            curv_note = (
                f"Min artery bend radius {Rmin:.1f} mm "
                f"({'>' if ok_curv else '<'} heuristic limit {thresh:.1f} mm = stent_length/2)"
            )
        else:
            curv_note = "Artery is straight — no curvature constraint"
            ok_curv   = True
        check_basis = "geometric heuristic  (set max_elastic_strain for material check)"

    checks["bending_strain"] = dict(
        max_curvature_per_mm      = round(kmax, 6),
        min_bend_radius_mm        = round(Rmin, 2) if np.isfinite(Rmin) else "inf",
        max_bending_strain_pct    = round(eps_actual * 100, 4) if eps_actual is not None else "N/A",
        elastic_limit_pct         = round(eps_max * 100, 2) if eps_max is not None else "N/A",
        bending_stiffness_EI_Nmm2 = round(EI, 4) if EI is not None else "N/A",
        check_basis               = check_basis,
        passed                    = ok_curv,
        note                      = curv_note,
    )

    return checks


# ── Run ────────────────────────────────────────────────────────────────────────
print("Running compatibility checks …")
print("  [1/5] Length      — inline")
print("  [2/5] Delivery    — inline")
report = check_stent_artery_fit(
    skel_mapped, skel, features, artery_mesh, artery_cl, artery_radius,
)
print("  [5/5] Bending strain — inline")

# ── Summary ────────────────────────────────────────────────────────────────────
_OK   = "\033[92m✓ PASS\033[0m"
_FAIL = "\033[91m✗ FAIL\033[0m"

print()
print("=" * 68)
print("  STENT – ARTERY  COMPATIBILITY REPORT")
print("=" * 68)

for name, r in report.items():
    tag = _OK if r["passed"] else _FAIL
    print(f"\n  {name.upper():<16s}  {tag}")
    print(f"  {'':<16s}  {r['note']}")
    if name == "bending_strain":
        print(f"  {'':<16s}  basis: {r['check_basis']}")
        if r['bending_stiffness_EI_Nmm2'] != "N/A":
            print(f"  {'':<16s}  EI = {r['bending_stiffness_EI_Nmm2']:.4g} N·mm²")

print()
print("=" * 68)
failed = [k for k, v in report.items() if not v["passed"]]
if not failed:
    print("  OVERALL  →  ALL CHECKS PASSED — proceed to simulation")
else:
    print(f"  OVERALL  →  FAILED: {', '.join(f.upper() for f in failed)}")
    print("              Resolve these issues before running simulation.")
print("=" * 68)

Running compatibility checks …
  [1/5] Length      — inline
  [2/5] Delivery    — inline
  [3/5] Containment + [4/5] Clearance … 100.0% inside, min clearance 0.477 mm, 0 penetrating
  [5/5] Bending strain — inline

  STENT – ARTERY  COMPATIBILITY REPORT

  LENGTH            ✓ PASS
                    Stent 18.09 mm fills 50.0% of 36.19 mm artery arc

  DELIVERY          ✓ PASS
                    Crimped OD 3.145 mm < artery ID 7.145 mm (56.0% radial margin)

  CONTAINMENT       ✓ PASS
                    100.0% of 3,000 sampled nodes inside artery lumen

  CLEARANCE         ✓ PASS
                    Min wall clearance 0.477 mm (strut radius 0.053 mm), 0 penetrating nodes

  BENDING_STRAIN    ✓ PASS
                    Max strut bending strain 0.356% at R_min=15.0 mm (< elastic limit 8.0%)
                    basis: material  [ε = r_strut/R,  limit = 8.0%]
                    EI = 0.3185 N·mm²

  OVERALL  →  ALL CHECKS PASSED — proceed to simulation


### Step 5 — 4C contact simulation input

Converts the placed stent skeleton and artery mesh into a 4C `.dat` input file for beam-to-solid contact simulation.

| Entity | Discretisation | Treatment |
|--------|---------------|-----------|
| Stent struts | `BEAM3R LINE2` (Reissner beam) | elastic, free to expand |
| Artery wall | `SHELL_REISSNER TRI3` | rigid first pass — all DOFs fixed |
| Contact | beam-to-solid, penalty method | stent = slave · artery = master |

Set the parameters in the next cell, then run the generator. The expansion load section is left as a documented TODO in the output file — choose the deployment strategy with your supervisor before running 4C.

In [95]:
# ─── Step 5 simulation parameters  (edit before running the generator) ────────
# Expansion: radial line load per paper (Ranno et al. 2025, Sci. Rep.)
RADIAL_LINE_LOAD  = 0.01e-3  # [N/mm]  = 0.01 N/m  (paper value, drives stent expansion)
N_LOAD_STEPS      = 100      # 100 quasi-static steps (paper value)
CONTACT_PENALTY   = 1.0e5   # beam-to-solid contact penalty  [N/mm²]
ARTERY_THICKNESS  = 0.5     # artery wall shell element thickness  [mm]

In [96]:
# ─── Step 5: Generate 4C .dat input file ─────────────────────────────────────
# Method: Ranno et al. 2025, Sci. Rep. — mixed-dimensional beam-to-solid contact
#   Stent  → BEAM3R LINE2  (geometrically exact Simo-Reissner beam)
#   Artery → SHELL_REISSNER TRI3  (rigid first pass — all DOFs fixed)
#   Load   → radial distributed line load 0.01 N/m (= RADIAL_LINE_LOAD in N/mm)
#             linearly ramped over N_LOAD_STEPS quasi-static steps
#   Contact → beam-to-solid penalty, stent = slave, artery = master
# Unit system: mm – N – tonne

# ── Node numbering (4C is 1-indexed) ─────────────────────────────────────────
# IDs 1 … n_sn             : stent skeleton nodes
# IDs n_sn+1 … n_sn+n_av  : artery surface nodes
n_sn          = len(skel_mapped)
artery_verts  = artery_mesh.vertices
artery_faces  = artery_mesh.faces
n_av          = len(artery_verts)
n_af          = len(artery_faces)
artery_offset = n_sn

# ── Unique beam elements ──────────────────────────────────────────────────────
_beam_elems = []
_seen_edges  = set()
for _, row in skel.iterrows():
    pid = int(row["skeleton_point_id"])
    for nid in row["neighbor_ids"]:
        if nid >= n_sn:
            continue
        key = (min(pid, nid), max(pid, nid))
        if key in _seen_edges:
            continue
        _seen_edges.add(key)
        _beam_elems.append((pid + 1, nid + 1))

# ── Cross-section & material ──────────────────────────────────────────────────
_r_s   = features["strut_thickness"] / 2.0
_A_cs  = np.pi * _r_s**2
_I_cs  = np.pi * _r_s**4 / 4.0
_Ip_cs = np.pi * _r_s**4 / 2.0
_kappa = 0.9
_E, _nu, _rho = features["youngs_modulus"], features["poissons_ratio"], features["density"]

# ── Boundary node groups ──────────────────────────────────────────────────────
_z_min_f, _z_max_f = features["z_min"], features["z_max"]
_t_stent   = (skel["z"].values - _z_min_f) / (_z_max_f - _z_min_f)
_end_mask   = (_t_stent <= 0.05) | (_t_stent >= 0.95)
_stent_end_gnids = (np.where(_end_mask)[0] + 1).tolist()
_stent_all_gnids = list(range(1, n_sn + 1))
_artery_all_gnids = list(range(artery_offset + 1, artery_offset + n_av + 1))

# ── Per-node radial expansion forces ─────────────────────────────────────────
# Paper: distributed line load 0.01 N/m radially outward (Ranno et al. 2025)
# Convert to per-node point loads via tributary length summation:
#   f_i = RADIAL_LINE_LOAD × tributary_length_i × radial_unit_vector_i
# Tributary length of node i = sum of half-lengths of all connected elements.
# Radial direction: from artery centreline to node  (cl_m available from cell 2.5)
_trib_len = np.zeros(n_sn)
for _n1, _n2 in _beam_elems:
    _elen = np.linalg.norm(skel_mapped[_n1 - 1] - skel_mapped[_n2 - 1])
    _trib_len[_n1 - 1] += _elen / 2.0
    _trib_len[_n2 - 1] += _elen / 2.0

_radial_vecs = skel_mapped - cl_m                                     # centreline → node
_radial_nrm  = np.linalg.norm(_radial_vecs, axis=1, keepdims=True)
_radial_hats = _radial_vecs / np.where(_radial_nrm > 1e-12, _radial_nrm, 1.0)
_node_forces = RADIAL_LINE_LOAD * _trib_len[:, None] * _radial_hats  # (N, 3) in [N]

# ── Quaternion helper ─────────────────────────────────────────────────────────
def _beam_quat(p1, p2):
    t = p2 - p1
    nrm = np.linalg.norm(t)
    if nrm < 1e-12:
        return 1.0, 0.0, 0.0, 0.0
    t /= nrm
    dot = float(np.clip(t[0], -1.0, 1.0))
    if dot > 1.0 - 1e-9:
        return 1.0, 0.0, 0.0, 0.0
    if dot < -1.0 + 1e-9:
        return 0.0, 0.0, 1.0, 0.0
    axis  = np.cross([1.0, 0.0, 0.0], t)
    axis /= np.linalg.norm(axis)
    a2    = np.arccos(dot) / 2.0
    s     = float(np.sin(a2))
    return float(np.cos(a2)), s * axis[0], s * axis[1], s * axis[2]

# ── Summary ───────────────────────────────────────────────────────────────────
_n_be        = len(_beam_elems)
_n_nodes_tot = n_sn + n_av
_n_elems_tot = _n_be + n_af

print("─── Step 5: 4C input file ───────────────────────────────────────────")
print(f"  Stent nodes       : {n_sn:>8,}   ({len(_stent_end_gnids)} end nodes fixed)")
print(f"  Artery vertices   : {n_av:>8,}")
print(f"  Beam elements     : {_n_be:>8,}   BEAM3R LINE2")
print(f"  Shell elements    : {n_af:>8,}   SHELL_REISSNER TRI3")
print(f"  Total nodes       : {_n_nodes_tot:>8,}   |   Total elements: {_n_elems_tot:,}")
print(f"  r_strut = {_r_s:.4f} mm  A = {_A_cs:.5f} mm²  I = {_I_cs:.6f} mm⁴")
print(f"  E = {_E:.0f} MPa  ν = {_nu}  ρ = {_rho:.2e} t/mm³")
print(f"  Expansion load    : {RADIAL_LINE_LOAD*1e3:.4f} N/m radially outward over {N_LOAD_STEPS} steps")
print(f"  Max nodal force   : {np.linalg.norm(_node_forces, axis=1).max():.4e} N")
print(f"  Total radial force: {np.linalg.norm(_node_forces.sum(axis=0)):.4e} N")

# ── Design entity IDs ─────────────────────────────────────────────────────────
_DSURF_ARTERY = 1   # all artery nodes → rigid BC + contact master
_DLINE_ENDS   = 1   # stent end nodes  → fix all rigid body modes
_DLINE_ALL    = 2   # all stent nodes  → contact slave

# ── Write .dat ────────────────────────────────────────────────────────────────
_out_dir  = pathlib.Path("../../notebook_outputs") / STENT_NAME / "4c_input"
_out_dir.mkdir(parents=True, exist_ok=True)
_dat_path = _out_dir / "stent_deployment.dat"

with open(_dat_path, "w") as _fh:
    def _w(s): _fh.write(s)

    _w("// ================================================================\n")
    _w(f"// 4C input — virtual stent deployment\n")
    _w(f"// Method: Ranno et al. 2025 (Sci. Rep.) — mixed-dimensional beam-to-solid contact\n")
    _w(f"// stent    : {STENT_NAME}   material : {MATERIAL_NAME}\n")
    _w(f"// artery   : {ARTERY_TYPE}\n")
    _w(f"// load     : {RADIAL_LINE_LOAD*1e3:.4f} N/m radially outward over {N_LOAD_STEPS} steps\n")
    _w(f"// nodes    : {n_sn} stent + {n_av} artery = {_n_nodes_tot} total\n")
    _w(f"// elements : {_n_be} BEAM3R + {n_af} TRI3 = {_n_elems_tot} total\n")
    _w("// unit system : mm – N – tonne\n")
    _w("// ================================================================\n\n")

    # PROBLEM TYP
    _w("--PROBLEM TYP\n")
    _w("PROBLEMTYPE                     Structure\n")
    _w("RESTART                         0\n\n")

    # IO
    _w("--IO\n")
    _w("OUTPUT_BIN                      Yes\n")
    _w("STRUCT_DISP                     Yes\n")
    _w("STRUCT_STRESS                   cauchy\n")
    _w("STRUCT_STRAIN                   gl\n")
    _w("FILESTEPS                       1\n")
    _w("STDOUTEVERY                     1\n\n")

    # TIME FUNCTION — linear ramp 0 → 1 over pseudo-time 0 → 1
    _w("--FUNCT1\n")
    _w("SYMBOLIC_FUNCTION_OF_TIME t\n\n")

    # STRUCTURAL DYNAMIC
    _w("--STRUCTURAL DYNAMIC\n")
    _w("INT_STRATEGY                    Standard\n")
    _w("DYNAMICTYPE                     Statics\n")
    _w("RESULTSEVERY                    1\n")
    _w("RESTARTEVERY                    10\n")
    _w("NLNSOL                          fullnewton\n")
    _w(f"NUMSTEP                         {N_LOAD_STEPS}\n")
    _w(f"TIMESTEP                        {1.0 / N_LOAD_STEPS:.8f}\n")
    _w("MAXTIME                         1.0\n")
    _w("TOLRES                          1.0e-8\n")
    _w("TOLDISP                         1.0e-8\n")
    _w("MAXITER                         50\n\n")

    # SOLVER
    _w("--SOLVER 1\n")
    _w("NAME                            Structure_Solver\n")
    _w("SOLVER                          Superlu\n\n")

    # BEAM INTERACTION
    _w("--BEAM INTERACTION/BEAM TO SOLID SURFACE CONTACT\n")
    _w("// TODO: verify section name and keywords against your 4C version\n")
    _w(f"PENALTY_PARAMETER               {CONTACT_PENALTY:.2e}\n")
    _w("INTEGRATION_SEGMENTS            5\n")
    _w("STRATEGY                        Penalty\n\n")

    # PROBLEM SIZE
    _w("--PROBLEM SIZE\n")
    _w(f"ELEMENTS                        {_n_elems_tot}\n")
    _w(f"NODES                           {_n_nodes_tot}\n\n")

    # MATERIALS
    _w("--MATERIALS\n")
    _w(f"// MAT 1 — stent strut: {MATERIAL_NAME}  r_strut = {_r_s:.4f} mm  (circular cross-section)\n")
    _w(
        f"MAT 1 MAT_BeamReissnerElastHyper"
        f" YOUNG {_E:.4f}"
        f" POISSONRATIO {_nu:.6f}"
        f" DENSITY {_rho:.6e}"
        f" CROSSSEC {_A_cs:.8f}"
        f" SHEARCORR {_kappa:.4f}"
        f" MOMINPOL {_Ip_cs:.10f}"
        f" MOMIN2 {_I_cs:.10f}"
        f" MOMIN3 {_I_cs:.10f}\n"
    )
    _w("// MAT 2 — artery wall (rigid first pass — all DOFs fixed via BC)\n")
    _w("//   Upgrade to HGO-C hyperelastic (Holzapfel-Gasser-Ogden) for deformable artery.\n")
    _w("//   Paper values (human coronary, Ranno et al. 2025):\n")
    _w("//     Media (0.4 mm): μ_matrix=0.06 MPa, fibers=0.112 MPa, exp.coeff=20.61, angle=41°\n")
    _w("//     Adventitia    : μ_matrix=0.024 MPa, fibers=0.362 MPa, exp.coeff=7.089, angle=50.1°\n")
    _w("MAT 2 MAT_Struct_StVenantKirchhoff YOUNG 1.0 NUE 0.45 DENS 1.0e-6\n\n")

    # NODE COORDS
    _w("--NODE COORDS\n")
    for _i, _pt in enumerate(skel_mapped):
        _w(f"NODE {_i+1:8d} COORD {_pt[0]:15.8f} {_pt[1]:15.8f} {_pt[2]:15.8f}\n")
    for _i, _pt in enumerate(artery_verts):
        _gid = artery_offset + _i + 1
        _w(f"NODE {_gid:8d} COORD {_pt[0]:15.8f} {_pt[1]:15.8f} {_pt[2]:15.8f}\n")
    _w("\n")

    # STRUCTURE ELEMENTS
    _w("--STRUCTURE ELEMENTS\n")
    _w("// BEAM3R LINE2 — geometrically exact Simo-Reissner beam (Ranno et al. 2025)\n")
    _w("// TRIADS: quaternion (w x y z); local x = beam tangent\n")
    _w("// TODO: verify TRIADS quaternion ordering (w,x,y,z vs x,y,z,w) for your 4C version\n")
    for _eid, (_n1, _n2) in enumerate(_beam_elems, start=1):
        _qw, _qx, _qy, _qz = _beam_quat(skel_mapped[_n1 - 1], skel_mapped[_n2 - 1])
        _w(
            f"{_eid:8d} BEAM3R LINE2 {_n1:8d} {_n2:8d} MAT 1"
            f" TRIADS {_qw:.6f} {_qx:.6f} {_qy:.6f} {_qz:.6f}"
            f"        {_qw:.6f} {_qx:.6f} {_qy:.6f} {_qz:.6f}\n"
        )
    _w("// SHELL_REISSNER TRI3 — artery wall (rigid; all DOFs fixed)\n")
    _w("// TODO: verify element keyword for your 4C version\n")
    for _fid, _face in enumerate(artery_faces):
        _gids = tuple(artery_offset + int(_face[_j]) + 1 for _j in range(3))
        _eid  = _n_be + _fid + 1
        _w(
            f"{_eid:8d} SHELL_REISSNER TRI3"
            f" {_gids[0]:8d} {_gids[1]:8d} {_gids[2]:8d}"
            f" MAT 2 KINEM nonlinear EAS none THICK {ARTERY_THICKNESS:.4f} SDC 1.0\n"
        )
    _w("\n")

    # DESIGN ENTITY TOPOLOGY
    _w("--DNODE-NODE TOPOLOGY\n")
    _w("// DNODE per stent node — needed for per-node Neumann (expansion) loads\n")
    for _gnid in _stent_all_gnids:
        _w(f"DNODE {_gnid} NODE {_gnid}\n")
    _w("\n")

    _w("--DSURF-NODE TOPOLOGY\n")
    _w(f"// DSURF {_DSURF_ARTERY}: all artery nodes (rigid wall BC + contact master)\n")
    for _gnid in _artery_all_gnids:
        _w(f"DSURF {_DSURF_ARTERY} NODE {_gnid}\n")
    _w("\n")

    _w("--DLINE-NODE TOPOLOGY\n")
    _w(f"// DLINE {_DLINE_ENDS}: stent end nodes — constrain all rigid body modes\n")
    for _gnid in _stent_end_gnids:
        _w(f"DLINE {_DLINE_ENDS} NODE {_gnid}\n")
    _w(f"// DLINE {_DLINE_ALL}: all stent nodes — contact slave side\n")
    for _gnid in _stent_all_gnids:
        _w(f"DLINE {_DLINE_ALL} NODE {_gnid}\n")
    _w("\n")

    # BOUNDARY CONDITIONS
    _w("// ── Dirichlet BCs ────────────────────────────────────────────────────\n")
    _w("// DOF order: Tx Ty Tz Rx Ry Rz  (1 = fixed, 0 = free)\n")
    _w("// TODO: verify section names and BC syntax for your 4C version\n\n")

    _w("--DESIGN SURF DIRICH CONDITIONS\n")
    _w(f"// Artery: fully fixed (rigid first pass)\n")
    _w(f"E {_DSURF_ARTERY} - 1 1 1 1 1 1  0.0 0.0 0.0 0.0 0.0 0.0  0 0 0 0 0 0\n\n")

    _w("--DESIGN LINE DIRICH CONDITIONS\n")
    _w(f"// Stent ends: fix Tz + all rotations — prevents rigid body motion\n")
    _w(f"// Radial DOFs (Tx, Ty) remain free so the stent can expand.\n")
    _w(f"E {_DLINE_ENDS} - 0 0 1 1 1 1  0.0 0.0 0.0 0.0 0.0 0.0  0 0 0 0 0 0\n\n")

    # NEUMANN BCs — radial expansion loads
    _w("// ── Neumann BCs: radial expansion load (Ranno et al. 2025) ──────────\n")
    _w(f"// Distributed line load: {RADIAL_LINE_LOAD*1e3:.4f} N/m radially outward\n")
    _w(f"// Converted to per-node point loads via tributary element length summation.\n")
    _w(f"// Ramped linearly 0→1 over {N_LOAD_STEPS} steps using FUNCT1 (t)\n")
    _w("// Format: E <dpoint_id> - <fx> <fy> <fz> <mx> <my> <mz> <funct_id>\n")
    _w("// TODO: verify DESIGN POINT NEUMANN syntax for your 4C version\n")
    _w("--DESIGN POINT NEUMANN CONDITIONS\n")
    for _i, _gnid in enumerate(_stent_all_gnids):
        _fx, _fy, _fz = _node_forces[_i]
        _w(f"E {_gnid} - {_fx:.8e} {_fy:.8e} {_fz:.8e} 0.0 0.0 0.0 1\n")
    _w("\n")

    # CONTACT PAIR
    _w("// ── Beam-to-solid contact pair ───────────────────────────────────────\n")
    _w(f"// Slave : DLINE {_DLINE_ALL} — stent beam elements\n")
    _w(f"// Master: DSURF {_DSURF_ARTERY} — artery wall shell surface\n")
    _w("// TODO: verify section name and pair syntax for your 4C version\n")
    _w("--BEAM TO SOLID SURFACE CONTACT CONDITIONS\n")
    _w(f"E {_DLINE_ALL} {_DSURF_ARTERY} Penalty {CONTACT_PENALTY:.2e}\n\n")

    _w("// ================================================================\n")
    _w("// END OF FILE\n")
    _w("// ================================================================\n")

_sz = _dat_path.stat().st_size
print(f"\nWritten: {_dat_path}")
print(f"  File size : {_sz / 1024 / 1024:.2f} MB  ({_sz:,} bytes)")

─── Step 5: 4C input file ───────────────────────────────────────────
  Stent nodes       :   20,968   (2198 end nodes fixed)
  Artery vertices   :    9,666
  Beam elements     :   21,166   BEAM3R LINE2
  Shell elements    :   19,328   SHELL_REISSNER TRI3
  Total nodes       :   30,634   |   Total elements: 40,494
  r_strut = 0.0534 mm  A = 0.00895 mm²  I = 0.000006 mm⁴
  E = 50000 MPa  ν = 0.33  ρ = 6.45e-09 t/mm³
  Expansion load    : 0.0100 N/m radially outward over 100 steps
  Max nodal force   : 2.3089e-07 N
  Total radial force: 6.0159e-06 N

Written: ../../notebook_outputs/stent2/4c_input/stent_deployment.dat
  File size : 9.13 MB  (9,569,026 bytes)
